**This is the baseline model its just 4 simple logistic regression models trained over the dataset from sklearn**

Results from this baseline is that while it gives higher accuracy it is awful at predicting failures as the dataset is extremely biased towards no failure

In [30]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [31]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
ai4i_2020_predictive_maintenance_dataset = fetch_ucirepo(id=601) 
  
# data (as pandas dataframes) 
X = ai4i_2020_predictive_maintenance_dataset.data.features 
y = ai4i_2020_predictive_maintenance_dataset.data.targets 
  
# metadata 
print(ai4i_2020_predictive_maintenance_dataset.metadata) 
  
# variable information 
print(ai4i_2020_predictive_maintenance_dataset.variables) 

{'uci_id': 601, 'name': 'AI4I 2020 Predictive Maintenance Dataset', 'repository_url': 'https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset', 'data_url': 'https://archive.ics.uci.edu/static/public/601/data.csv', 'abstract': 'The AI4I 2020 Predictive Maintenance Dataset is a synthetic dataset that reflects real predictive maintenance data encountered in industry.', 'area': 'Computer Science', 'tasks': ['Classification', 'Regression', 'Causal-Discovery'], 'characteristics': ['Multivariate', 'Time-Series'], 'num_instances': 10000, 'num_features': 6, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'], 'index_col': ['UID', 'Product ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2020, 'last_updated': 'Wed Feb 14 2024', 'dataset_doi': '10.24432/C5HS5C', 'creators': [], 'intro_paper': {'ID': 386, 'type': 'NATIVE', 'title': 'Explainable Artificial Intelligen

In [32]:
X.head()

,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear
0,M,298.1,308.6,1551,42.8,0
1,L,298.2,308.7,1408,46.3,3
2,L,298.1,308.5,1498,49.4,5
3,L,298.2,308.6,1433,39.5,7
4,L,298.2,308.7,1408,40.0,9


In [33]:
y.head()

,Machine failure,TWF,HDF,PWF,OSF,RNF
0,0,0,0,0,0,0
1,0,0,0,0,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0


In [34]:
y.nunique()

Machine failure    2
TWF                2
HDF                2
PWF                2
OSF                2
RNF                2
dtype: int64

In [35]:
X['Type'] = X['Type'].map({'M': 0, 'L': 1, 'H': 2})

C:\Users\MSI-Admin\AppData\Local\Temp\ipykernel_22224\963275165.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Type'] = X['Type'].map({'M': 0, 'L': 1, 'H': 2})


In [36]:
X.head()

,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear
0,0,298.1,308.6,1551,42.8,0
1,1,298.2,308.7,1408,46.3,3
2,1,298.1,308.5,1498,49.4,5
3,1,298.2,308.6,1433,39.5,7
4,1,298.2,308.7,1408,40.0,9


In [37]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import numpy as np
import pandas as pd


def create_balanced_training_set(X_train, y_train_target):
    train_data = X_train.copy()
    train_data['target'] = y_train_target.values

    majority_class = train_data[train_data['target'] == 0]
    minority_class = train_data[train_data['target'] == 1]

    minority_oversampled = resample(
        minority_class,
        replace=True,
        n_samples=len(majority_class),
        random_state=42
    )

    balanced_train_data = pd.concat([majority_class, minority_oversampled])
    balanced_train_data = balanced_train_data.sample(frac=1, random_state=42).reset_index(drop=True)

    X_train_balanced = balanced_train_data.drop(columns=['target'])
    y_train_balanced = balanced_train_data['target']

    return X_train_balanced, y_train_balanced


def train_evaluate_logistic_regression(target_column):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y[target_column]
    )

    X_train_balanced, y_train_balanced = create_balanced_training_set(X_train, y_train[target_column])

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_balanced)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression()
    model.fit(X_train_scaled, y_train_balanced)
    y_pred = model.predict(X_test_scaled)

    print(f"Target: {target_column}")
    print("Balanced training distribution:")
    print(y_train_balanced.value_counts())
    print(classification_report(y_test[target_column], y_pred))
    print(confusion_matrix(y_test[target_column], y_pred))


train_evaluate_logistic_regression('Machine failure')


Target: Machine failure
Balanced training distribution:
target
0    7729
1    7729
Name: count, dtype: int64
              precision    recall  f1-score   support

           0       0.99      0.82      0.90      1932
           1       0.14      0.84      0.24        68

    accuracy                           0.82      2000
   macro avg       0.57      0.83      0.57      2000
weighted avg       0.96      0.82      0.88      2000

[[1582  350]
 [  11   57]]


In [ ]:
train_evaluate_logistic_regression('TWF')

Target: TWF
Balanced training distribution:
target
0    7963
1    7963
Name: count, dtype: int64
              precision    recall  f1-score   support

           0       1.00      0.90      0.95      1991
           1       0.04      1.00      0.08         9

    accuracy                           0.90      2000
   macro avg       0.52      0.95      0.51      2000
weighted avg       1.00      0.90      0.94      2000

[[1791  200]
 [   0    9]]


In [39]:
train_evaluate_logistic_regression('HDF')


Target: HDF
Balanced training distribution:
target
1    7908
0    7908
Name: count, dtype: int64
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      1977
           1       0.35      1.00      0.52        23

    accuracy                           0.98      2000
   macro avg       0.68      0.99      0.76      2000
weighted avg       0.99      0.98      0.98      2000

[[1935   42]
 [   0   23]]


In [40]:
train_evaluate_logistic_regression('PWF')


Target: PWF
Balanced training distribution:
target
1    7924
0    7924
Name: count, dtype: int64
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      1981
           1       0.29      1.00      0.45        19

    accuracy                           0.98      2000
   macro avg       0.65      0.99      0.72      2000
weighted avg       0.99      0.98      0.98      2000

[[1935   46]
 [   0   19]]


In [41]:
train_evaluate_logistic_regression('OSF')


Target: OSF
Balanced training distribution:
target
0    7922
1    7922
Name: count, dtype: int64
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      1980
           1       0.49      1.00      0.66        20

    accuracy                           0.99      2000
   macro avg       0.74      0.99      0.83      2000
weighted avg       0.99      0.99      0.99      2000

[[1959   21]
 [   0   20]]


In [42]:
train_evaluate_logistic_regression('RNF')


Target: RNF
Balanced training distribution:
target
1    7985
0    7985
Name: count, dtype: int64
              precision    recall  f1-score   support

           0       1.00      0.62      0.76      1996
           1       0.00      0.75      0.01         4

    accuracy                           0.62      2000
   macro avg       0.50      0.68      0.39      2000
weighted avg       1.00      0.62      0.76      2000

[[1231  765]
 [   1    3]]


In [43]:
# Legacy duplicate cell removed. RNF is already evaluated above.


In [44]:
def cross_validate_logistic_regression(target_column, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    metrics_per_fold = []
    confusion_matrices = []

    for fold_number, (train_idx, test_idx) in enumerate(skf.split(X, y[target_column]), start=1):
        X_train_fold = X.iloc[train_idx].copy()
        X_test_fold = X.iloc[test_idx].copy()
        y_train_fold = y.iloc[train_idx][target_column]
        y_test_fold = y.iloc[test_idx][target_column]

        X_train_balanced, y_train_balanced = create_balanced_training_set(X_train_fold, y_train_fold)

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_balanced)
        X_test_scaled = scaler.transform(X_test_fold)

        model = LogisticRegression()
        model.fit(X_train_scaled, y_train_balanced)

        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]

        fold_metrics = {
            'fold': fold_number,
            'accuracy': accuracy_score(y_test_fold, y_pred),
            'precision': precision_score(y_test_fold, y_pred, zero_division=0),
            'recall': recall_score(y_test_fold, y_pred, zero_division=0),
            'f1': f1_score(y_test_fold, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_test_fold, y_prob)
        }
        metrics_per_fold.append(fold_metrics)
        confusion_matrices.append(confusion_matrix(y_test_fold, y_pred))

    metrics_df = pd.DataFrame(metrics_per_fold)
    print(f"Cross-validation results for: {target_column}")
    print(metrics_df)
    print()
    print('Mean metrics across folds:')
    print(metrics_df.drop(columns=['fold']).mean().round(4))
    print()
    print('Standard deviation across folds:')
    print(metrics_df.drop(columns=['fold']).std().round(4))
    print()
    print('Summed confusion matrix across folds:')
    print(np.sum(confusion_matrices, axis=0))


for target in ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']:
    cross_validate_logistic_regression(target)
    print('=' * 60)


Cross-validation results for: Machine failure
   fold  accuracy  precision    recall        f1   roc_auc
0     1    0.8205   0.133166  0.791045  0.227957  0.894310
1     2    0.8240   0.139594  0.808824  0.238095  0.892241
2     3    0.8090   0.133178  0.838235  0.229839  0.911521
3     4    0.8205   0.144254  0.867647  0.247379  0.911475
4     5    0.8320   0.137838  0.750000  0.232877  0.887460

Mean metrics across folds:
accuracy     0.8212
precision    0.1376
recall       0.8112
f1           0.2352
roc_auc      0.8994
dtype: float64

Standard deviation across folds:
accuracy     0.0083
precision    0.0047
recall       0.0449
f1           0.0078
roc_auc      0.0113
dtype: float64

Summed confusion matrix across folds:
[[7937 1724]
 [  64  275]]
Cross-validation results for: TWF
   fold  accuracy  precision    recall        f1   roc_auc
0     1    0.9135   0.039326  0.777778  0.074866  0.959763
1     2    0.9165   0.051136  1.000000  0.097297  0.979519
2     3    0.9040   0.044776  1